# Notebook 1: Data Exploration
**Project:** Secure & Explainable Fraud Detection System  
**Authors:** Anis Mohammed Metkari & Sharwari Rajendra Nimbalkar  
**Supervisor:** Irina Tal – Dublin City University

## Objectives
- Load and profile the fraud dataset
- Analyse class imbalance
- Visualise feature distributions (fraud vs. legitimate)
- Identify key correlations

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from utils.data_loader import DataLoader, LABEL_COLUMN

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1. Load Dataset
If `creditcard.csv` is available in `data/`, set the path below.  
Otherwise synthetic data will be generated automatically.

In [ ]:
DATA_PATH = '../data/creditcard.csv'  # change or set to None to use synthetic data
loader = DataLoader(data_path=DATA_PATH if os.path.exists(DATA_PATH) else None)
df = loader.load()
print(f'Shape: {df.shape}')
df.head()

## 2. Class Distribution

In [ ]:
counts = df[LABEL_COLUMN].value_counts()
print(counts)
print(f'Fraud rate: {counts[1]/len(df)*100:.2f}%')

fig, ax = plt.subplots(figsize=(5, 4))
counts.plot(kind='bar', ax=ax, color=['steelblue','firebrick'])
ax.set_xticklabels(['Legitimate', 'Fraud'], rotation=0)
ax.set_title('Class Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('../outputs/class_distribution.png', dpi=150)
plt.show()

## 3. Amount Distribution by Class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for cls, color, label in [(0,'steelblue','Legitimate'), (1,'firebrick','Fraud')]:
    sub = df[df[LABEL_COLUMN] == cls]['amount']
    axes[0].hist(sub, bins=50, alpha=0.6, label=label, color=color)
    axes[1].hist(sub.clip(upper=sub.quantile(0.95)), bins=50, alpha=0.6, label=label, color=color)

axes[0].set_title('Amount Distribution (full range)')
axes[1].set_title('Amount Distribution (95th percentile clip)')
for ax in axes:
    ax.set_xlabel('Amount')
    ax.set_ylabel('Count')
    ax.legend()
plt.tight_layout()
plt.savefig('../outputs/amount_distribution.png', dpi=150)
plt.show()

## 4. Categorical Feature Analysis

In [ ]:
cat_cols = ['merchant_category', 'device_type']
for col in cat_cols:
    if col not in df.columns:
        continue
    fraud_rate = df.groupby(col)[LABEL_COLUMN].mean().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(8, 3))
    fraud_rate.plot(kind='bar', ax=ax, color='coral')
    ax.set_title(f'Fraud Rate by {col}')
    ax.set_ylabel('Fraud Rate')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(f'../outputs/fraud_rate_{col}.png', dpi=150)
    plt.show()

## 5. Correlation Matrix (numeric features)

In [ ]:
numeric_df = df.select_dtypes(include='number')
corr = numeric_df.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', center=0, fmt='.2f', annot=len(numeric_df.columns) <= 10)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../outputs/correlation_matrix.png', dpi=150)
plt.show()

## 6. Summary Statistics

In [ ]:
print('=== Fraud Transactions ===')
display(df[df[LABEL_COLUMN] == 1].describe())
print('\n=== Legitimate Transactions ===')
display(df[df[LABEL_COLUMN] == 0].describe())